# Notebook 05: Solar Farm Detection — Fine-Tuning

Compares the rule-based NDBI/NDVI detector against the fine-tuned MobileNetV3-small CNN.

Solar panels have distinctive spectral properties:
- Low NDVI (non-vegetated)
- Elevated NDBI (reflective surfaces)
- Very uniform texture within panel arrays
- Sharp geometric boundaries

In [ ]:
import sys; sys.path.insert(0, '..')
from dotenv import load_dotenv; load_dotenv('../.env')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

from apps.api.services.imagery import SentinelHubFetcher
from apps.api.services.change.solar import SolarDetector
from apps.api.services.change.generic import compute_ndvi, compute_ndbi

In [ ]:
# Tumkur Solar Corridor — known large-scale solar parks
BBOX = [76.9, 13.2, 77.4, 13.7]

fetcher = SentinelHubFetcher()
before = fetcher.get_sentinel2_composite(BBOX, '2019-01-01', '2019-03-31')  # pre-solar
after  = fetcher.get_sentinel2_composite(BBOX, '2024-01-01', '2024-03-31')  # post-installation

print(f'Imagery loaded: {before.shape}')

In [ ]:
# Compute spectral indices for before/after
ndvi_before = compute_ndvi(before)
ndvi_after  = compute_ndvi(after)
ndbi_before = compute_ndbi(before)
ndbi_after  = compute_ndbi(after)

ndvi_delta = ndvi_after - ndvi_before
ndbi_delta = ndbi_after - ndbi_before

print('NDVI delta stats:')
print(f'  Mean={ndvi_delta.mean():.4f}  Std={ndvi_delta.std():.4f}')
print('NDBI delta stats:')
print(f'  Mean={ndbi_delta.mean():.4f}  Std={ndbi_delta.std():.4f}')

In [ ]:
# Rule-based solar detection
detector = SolarDetector(ndbi_threshold=0.1, ndvi_threshold=0.15)
mask_rule, conf_rule = detector.detect(before, after)

print(f'Rule-based: {mask_rule.sum():,} pixels flagged ({100*mask_rule.mean():.2f}%)')
print(f'Confidence: {conf_rule:.3f}')

In [ ]:
# Visualisation
def to_rgb(arr):
    rgb = arr[:, :, [3, 2, 1]]
    return np.power(np.clip(rgb, 0, 0.3) / 0.3, 0.4)

fig, axes = plt.subplots(2, 3, figsize=(15, 10))

axes[0, 0].imshow(to_rgb(before)); axes[0, 0].set_title('Before (2019)'); axes[0, 0].axis('off')
axes[0, 1].imshow(to_rgb(after));  axes[0, 1].set_title('After (2024)');  axes[0, 1].axis('off')

im = axes[0, 2].imshow(ndbi_delta, cmap='RdYlBu_r', vmin=-0.3, vmax=0.3)
axes[0, 2].set_title('NDBI Delta'); axes[0, 2].axis('off')
fig.colorbar(im, ax=axes[0, 2], fraction=0.046)

im2 = axes[1, 0].imshow(ndvi_delta, cmap='RdYlGn', vmin=-0.5, vmax=0.5)
axes[1, 0].set_title('NDVI Delta'); axes[1, 0].axis('off')
fig.colorbar(im2, ax=axes[1, 0], fraction=0.046)

axes[1, 1].imshow(to_rgb(after))
axes[1, 1].imshow(mask_rule, cmap='Reds', alpha=0.6)
axes[1, 1].set_title(f'Rule-based Detection (conf={conf_rule:.2f})')
axes[1, 1].axis('off')

# NDBI vs NDVI scatter of changed pixels
changed_ndbi = ndbi_delta[mask_rule]
changed_ndvi = ndvi_delta[mask_rule]
unchanged_ndbi = ndbi_delta[~mask_rule].ravel()[::50]  # subsample
unchanged_ndvi = ndvi_delta[~mask_rule].ravel()[::50]

axes[1, 2].scatter(unchanged_ndbi, unchanged_ndvi, c='gray', s=1, alpha=0.2, label='No change')
axes[1, 2].scatter(changed_ndbi, changed_ndvi, c='red', s=3, alpha=0.5, label='Solar detected')
axes[1, 2].axhline(0, color='k', linewidth=0.5)
axes[1, 2].axvline(0, color='k', linewidth=0.5)
axes[1, 2].set_xlabel('NDBI delta')
axes[1, 2].set_ylabel('NDVI delta')
axes[1, 2].set_title('Feature Space (Solar quadrant: +NDBI, -NDVI)')
axes[1, 2].legend(markerscale=5)

plt.suptitle('Solar Farm Detection — Tumkur', fontsize=12, fontweight='bold')
plt.tight_layout()
plt.savefig('05_solar_detection.png', dpi=150)
plt.show()

In [ ]:
# Check if CNN model is available
from pathlib import Path
model_path = Path('../models/solar_current.pt')

if model_path.exists():
    print(f'CNN model found at {model_path} — CNN detection will be used')
    mask_cnn, conf_cnn = detector.detect(before, after)  # SolarDetector auto-uses CNN
    print(f'CNN detection: {mask_cnn.sum():,} pixels, confidence={conf_cnn:.3f}')
    
    # Compare rule vs CNN
    agreement = (mask_rule == mask_cnn).mean()
    print(f'Agreement between rule-based and CNN: {100*agreement:.1f}%')
else:
    print(f'No CNN model at {model_path}')
    print('Run: python ml/train_solar.py --epochs 20')